In [9]:
import pandas as pd

In [10]:
# Path to GLSEA SST data
ver_file = '/Users/ljob/Desktop/CNBS_forecast_ver_combined.csv'
bias_file = '/Users/ljob/Desktop/CNBS_forecast_ver_sst_combined.csv'

ver_data = pd.read_csv(ver_file,sep='\t')
bias_data = pd.read_csv(bias_file,sep='\t')


In [11]:
import pandas as pd
import numpy as np

# ----------------------------
# CONFIG
# ----------------------------
META_COLS = ["cfs_run", "model", "forecast_month"]

# ----------------------------
# METRIC FUNCTION
# ----------------------------
def compute_skill_grouped(df, metric="rmse"):
    if metric == "rmse":
        return np.sqrt(np.mean((df["obs"] - df["forecast"])**2))
    elif metric == "mae":
        return np.mean(np.abs(df["obs"] - df["forecast"]))
    elif metric == "corr":
        return df["obs"].corr(df["forecast"])
    else:
        raise ValueError("Unsupported metric")


# ----------------------------
# MELT + STRUCTURE
# ----------------------------
def reshape_long(df):
    df = df.copy()

    # normalize time
    df["forecast_month"] = pd.to_datetime(df["forecast_month"])

    value_cols = [c for c in df.columns if c not in META_COLS]

    long_df = df.melt(
        id_vars=META_COLS,
        value_vars=value_cols,
        var_name="variable",
        value_name="value"
    )

    # identify obs vs forecast
    long_df["type"] = np.where(
        long_df["variable"].str.endswith("_obs"),
        "obs",
        "forecast"
    )

    # strip _obs
    long_df["variable_clean"] = long_df["variable"].str.replace("_obs", "", regex=False)

    # split lake + component
    split = long_df["variable_clean"].str.split("_", n=1, expand=True)
    long_df["lake"] = split[0]
    long_df["component"] = split[1]

    # pivot so forecast + obs are side-by-side
    wide = long_df.pivot_table(
        index=["forecast_month", "lake", "component", "cfs_run"],
        columns="type",
        values="value"
    ).reset_index()

    return wide


# ----------------------------
# COMPUTE SKILL
# ----------------------------
def compute_skill(df_long, metric="rmse"):
    skill = (
        df_long
        .dropna(subset=["forecast", "obs"])
        .groupby(["forecast_month", "lake", "component"])
        .apply(lambda g: compute_skill_grouped(g, metric))
        .reset_index(name="skill")
    )

    return skill


# ----------------------------
# COMPARE DATASETS
# ----------------------------
def compare_datasets(df1, df2, metric="rmse"):
    long1 = reshape_long(df1)
    long2 = reshape_long(df2)

    skill1 = compute_skill(long1, metric).rename(columns={"skill": "skill_A"})
    skill2 = compute_skill(long2, metric).rename(columns={"skill": "skill_B"})

    merged = pd.merge(
        skill1,
        skill2,
        on=["forecast_month", "lake", "component"],
        how="inner"
    )

    # improvement logic
    if metric in ["rmse", "mae"]:
        merged["improvement_pct"] = (
            (merged["skill_A"] - merged["skill_B"]) / merged["skill_A"] * 100
        )
    else:
        merged["improvement_pct"] = (
            (merged["skill_B"] - merged["skill_A"]) / np.abs(merged["skill_A"]) * 100
        )

    merged["winner"] = np.where(
        merged["improvement_pct"] > 0,
        "Dataset B",
        "Dataset A"
    )

    return merged


# ----------------------------
# SUMMARY
# ----------------------------
def summarize_results(comparison_df):
    return (
        comparison_df
        .groupby(["lake", "component"])["improvement_pct"]
        .mean()
        .reset_index()
        .sort_values("improvement_pct", ascending=False)
    )


# ----------------------------
# USAGE
# ----------------------------
comparison = compare_datasets(ver_data, bias_data, metric="rmse")

summary = summarize_results(comparison)

print("\n=== Detailed Comparison ===")
print(comparison.head())

print("\n=== Summary ===")
print(summary)

/var/folders/2w/ddc7n0594ydfswtzw_mmfs6c0000gp/T/ipykernel_67952/1415511733.py:74: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: compute_skill_grouped(g, metric))



=== Detailed Comparison ===
  forecast_month            lake      component     skill_A     skill_B  \
0     2011-04-01            erie    evaporation   30.608085   29.868223   
1     2011-04-01            erie            nbs  188.880266  124.673408   
2     2011-04-01            erie  precipitation   34.556030   30.591053   
3     2011-04-01            erie         runoff   58.555487   30.359060   
4     2011-04-01  michigan-huron    evaporation   23.668318   24.524170   

   improvement_pct     winner  
0         2.417211  Dataset B  
1        33.993418  Dataset B  
2        11.474052  Dataset B  
3        48.153347  Dataset B  
4        -3.616025  Dataset A  

=== Summary ===
              lake      component  improvement_pct
0             erie    evaporation        -1.972882
2             erie  precipitation        -3.253863
6   michigan-huron  precipitation        -4.420350
10         ontario  precipitation        -4.618121
5   michigan-huron            nbs        -5.573445
1    

/var/folders/2w/ddc7n0594ydfswtzw_mmfs6c0000gp/T/ipykernel_67952/1415511733.py:74: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: compute_skill_grouped(g, metric))
